[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 困难：旋转位置编码（RoPE）

实现 **RoPE** — 用于 LLaMA、GPT-NeoX 和大多数现代 LLM 的位置编码。在 **q, k** 上添上 RoPE 位置编码。

### 函数签名
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D)，其中 D 为偶数
    # 返回相同形状的旋转后 (q, k)
```

### 核心思想
将每个向量拆分为连续的对。按 `θ = pos / 10000^(2i/D)` 旋转每对：
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
这使得 `dot(q_rot[i], k_rot[j])` 仅依赖于 `i - j`（相对位置）。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import math

In [ ]:
# ✏️ 在此实现你的代码

def apply_rope(q, k):
    # 1. Compute position angles
    # 2. Split into even/odd pairs
    # 3. Apply rotation
    pass

1. **维度要求**：D必须为偶数，因为需要将向量拆分为连续的二维对。

2. **角度计算**：θ = 1 / 10000^(2i/D)，其中i从0到D/2-1。实现中使用指数形式避免数值溢出。

3. **旋转矩阵**：对每个二维对[x₀, x₁]应用旋转：
   - x₀' = x₀·cos(θ) - x₁·sin(θ)
   - x₁' = x₀·sin(θ) + x₁·cos(θ)

4. **相对位置编码**：经过RoPE后，q[i]和k[j]的点积只依赖于相对位置(i-j)，这使得模型能够更好地捕捉位置关系。

- 数学推导:\
$\theta_i = \frac{1}{10000^{2i/D}}$ 对于任何正数a，有：$a^b = e^{b \cdot \ln(a)}$ 所以： $10000^{2i/D} = e^{(2i/D) \cdot \ln(10000)}$; $\theta_i = \frac{1}{10000^{2i/D}} = \frac{1}{e^{(2i/D) \cdot \ln(10000)}}$; $\frac{1}{e^x} = e^{-x}$ 因此：$\theta_i = e^{-(2i/D) \cdot \ln(10000)}$

- 为什么使用指数形式？

1. **数值稳定性**：当 `D` 很大时，`2*i/D` 可能很小，直接用幂运算可能导致精度损失。指数形式更稳定。

2. **计算效率**：`torch.exp` 和 `math.log` 是高度优化的操作。

3. **避免溢出**：对于大的 `i` 值，`10000 ** (2*i/D)` 可能变得极大，而指数形式通过负指数避免了大数运算。

所以代码中的 `math.log(10000.0) / (D / 2)` 实际上就是 `2 * ln(10000) / D`，这样 `theta = exp(-i * 2 * ln(10000) / D)`，完全等价于原始公式。

In [ ]:
def apply_rope(q: torch.Tensor, k: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """
    对q和k应用旋转位置编码（RoPE）
    
    Args:
        q: (B, S, D) 查询张量，D为偶数
        k: (B, S, D) 键张量，D为偶数
    
    Returns:
        旋转后的(q, k)元组，形状不变
    """
    B, S, D = q.shape
    
    # 确保D为偶数
    assert D % 2 == 0, f"Dimension D must be even, got {D}"
    
    # 获取设备
    device = q.device
    
    # 生成位置索引 (S,)
    positions = torch.arange(S, device=device).float()  # (S,)
    
    # 生成维度索引 (D//2,)
    dim_indices = torch.arange(D // 2, device=device).float()  # (D//2,)
    
    # 计算theta: θ = 1 / 10000^(2i/D)
    # 使用指数形式: θ = exp(-2i/D * ln(10000)), 其中代码中 dim_indices / (D / 2) 写法表示 indices 在维度的相对位置，间隔为 2
    theta = torch.exp(- dim_indices * (math.log(10000.0) / (D / 2)))  # (D//2,)
    
    # 计算角度: angle = pos * theta
    # 形状: (S, D//2)
    angles = positions.unsqueeze(1) * theta.unsqueeze(0)  # (S, D//2), positions.unsqueeze(1) 表示在第一维度多加一个维度, (S,) -> (1, S)
    
    # 计算cos和sin
    cos = torch.cos(angles)  # (S, D//2)
    sin = torch.sin(angles)  # (S, D//2)
    
    # 将 q 和 k 改成 (B, S, D//2, 2) 的形状，最后一维为每对元素
    q_reshaped = q.view(B, S, D // 2, 2)
    k_reshaped = k.view(B, S, D // 2, 2)
    
    # 提取 x0 和 x1 (每对中的两个元素)
    q_x0 = q_reshaped[..., 0]  # (B, S, D//2)
    q_x1 = q_reshaped[..., 1]  # (B, S, D//2)
    k_x0 = k_reshaped[..., 0]  # (B, S, D//2)
    k_x1 = k_reshaped[..., 1]  # (B, S, D//2)
    
    # 扩展cos和sin到批次维度
    cos_exp = cos.unsqueeze(0)  # (1, S, D//2)
    sin_exp = sin.unsqueeze(0)  # (1, S, D//2)
    
    # 应用旋转矩阵:
    # [x0'] = [cos  -sin] [x0]
    # [x1']   [sin   cos] [x1]
    q_rotated_0 = q_x0 * cos_exp - q_x1 * sin_exp  # (B, S, D//2)
    q_rotated_1 = q_x0 * sin_exp + q_x1 * cos_exp  # (B, S, D//2)
    k_rotated_0 = k_x0 * cos_exp - k_x1 * sin_exp  # (B, S, D//2)
    k_rotated_1 = k_x0 * sin_exp + k_x1 * cos_exp  # (B, S, D//2)
    
    # 组合回去， torch.stack 的功能是沿新维度拼接一系列张量
    q_rotated = torch.stack([q_rotated_0, q_rotated_1], dim=-1)  # (B, S, D//2, 2)
    k_rotated = torch.stack([k_rotated_0, k_rotated_1], dim=-1)  # (B, S, D//2, 2)
    
    # 展平为原始形状
    q_out = q_rotated.view(B, S, D)  # (B, S, D)
    k_out = k_rotated.view(B, S, D)  # (B, S, D)
    
    return q_out, k_out

In [ ]:
# 🧪 调试
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('形状保持:', qr.shape == q.shape)
print('范数保持:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

In [ ]:
# ✅ 提交
from torch_judge import check
check('rope')